# Step 8: Web app hello world

![Step 8 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/08-step.png)

**Goal:** start a tiny web server you can open in a browser.

FastAPI is the easiest Python web framework for this. We don't need HTML — just two endpoints.

We *demo* FastAPI here. The canonical `create_app` factory gets exported in step 9, where we'll add the gallery.

## Step 8.0 — Setup

In [ ]:
from pathlib import Path

import yaml

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (root for root in ROOT_CANDIDATES if (root / "tutorials").exists()),
    Path.cwd(),
)
CONFIG_FILE = PROJECT_ROOT / "config.yaml"

CONFIG = {}
if CONFIG_FILE.exists():
    CONFIG = yaml.safe_load(CONFIG_FILE.read_text()) or {}
    if not isinstance(CONFIG, dict):
        raise TypeError(f"{CONFIG_FILE} must contain a top-level mapping")
else:
    print(f"Config file not found at {CONFIG_FILE}; using defaults.")

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = PROJECT_ROOT / "tutorials"
DATA_FOLDER = PROJECT_ROOT / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = PROJECT_ROOT / "yolov8n.pt"

# From config.yaml (or hardcoded fallback)
PHONE_IP = str(CONFIG.get("phone_ip", "192.168.1.42"))
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = str(CONFIG.get("slack_webhook", ""))
HUGGINGFACE_API_KEY = str(CONFIG.get("huggingface_api_key", ""))

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Config file: {CONFIG_FILE}")
print(f"Phone URL: {PHONE_URL}")


## Step 8.1 — Hello world in 10 lines

An `app` object, a `@app.get("/")` decorator, a function that returns a dict. FastAPI turns the dict into JSON automatically.

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Bird Watcher", version="0.0.0")


@app.get("/")
def index() -> dict:
    """Landing page."""
    return {"app": "bird-watcher", "status": "watching"}


@app.get("/health")
def health() -> dict:
    """Health check — useful for uptime monitoring."""
    return {"status": "ok"}

## Step 8.2 — Test without starting a server

`TestClient` lets you hit your endpoints as if they were live. No `uvicorn` needed.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
r = client.get("/")
print(r.status_code, r.json())
assert r.status_code == 200
assert r.json() == {"app": "bird-watcher", "status": "watching"}

r = client.get("/health")
print(r.status_code, r.json())
assert r.status_code == 200
assert r.json() == {"status": "ok"}
print("✅ / and /health both respond")

## Step 8.3 — Why a factory function?

For a real app, you want to *configure* it at startup — where's the database? where are the snapshots? A factory takes those as args and returns an `app`:

```python
def create_app(db_file, snapshot_folder) -> FastAPI:
    app = FastAPI(...)
    # ... add endpoints that use db_file + snapshot_folder ...
    return app
```

We'll wire that up next, where the factory gets the `/gallery` endpoint and mounts `/snapshots` for thumbnails.

## What's next

**Step 9:** open [09-gallery.ipynb](09-gallery.ipynb) — we'll define the `create_app` factory, add a `/gallery` endpoint, and mount the snapshot folder.